# Traffic Simulation with Adaptive Signal Control
**Course:** CSE 413 - Simulation and Modeling Lab

**Goal:** Simulate a 4-way intersection and show whether an *Adaptive* signal gives lower waiting time than a *Fixed-time* signal.

**Steps:**
1. Load the dataset (UCI Metro Interstate Traffic Volume, with a synthetic fallback if unavailable)
2. Derive the arrival rate (lambda) from hourly traffic
3. Validate the arrival model (Chi-square + KS test)
4. Simulation model (Fixed vs Adaptive)
5. Monte Carlo (30 replications) - peak and off-peak
6. Two-sample t-test + confidence interval
7. Graphs + conclusion

## Step 0: Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

np.random.seed(123)  # reproducible results
os.makedirs("results", exist_ok=True)   # all outputs are saved here

## Step 1: Load the dataset
UCI dataset: *Metro Interstate Traffic Volume* (the `date_time` and `traffic_volume` columns are required).

- If you have internet access in Colab, it is downloaded automatically.
- Otherwise, upload `Metro_Interstate_Traffic_Volume.csv` manually and place it in the same folder.
- If neither works, **synthetic** data (mimicking the real pattern: morning/evening peaks) is used, so the notebook can always run.

In [ ]:
import io, gzip, zipfile, urllib.request

def load_uci():
    # 1) local csv
    for name in ["Metro_Interstate_Traffic_Volume.csv", "Metro_Interstate_Traffic_Volume.csv.gz"]:
        if os.path.exists(name):
            return pd.read_csv(name)
    # 2) download from UCI
    url = "https://archive.ics.uci.edu/static/public/492/metro+interstate+traffic+volume.zip"
    raw = urllib.request.urlopen(url, timeout=20).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    fname = z.namelist()[0]
    data = z.read(fname)
    if fname.endswith(".gz"):
        data = gzip.decompress(data)
    return pd.read_csv(io.BytesIO(data))

def make_synthetic(days=365):
    # hourly volume with 2 rush-hour peaks (7-8am, 4-6pm) + noise
    rng = np.random.default_rng(1)
    t = pd.date_range("2017-01-01", periods=days*24, freq="h")
    h = t.hour.values
    base = 800 + 5200*np.exp(-0.5*((h-7.5)/1.5)**2) + 5800*np.exp(-0.5*((h-16.5)/1.8)**2) + 1500*np.exp(-0.5*((h-12)/3)**2)
    vol = base * rng.normal(1, 0.08, len(t))
    return pd.DataFrame({"date_time": t, "traffic_volume": vol.clip(50).astype(int)})

try:
    df = load_uci()
    SOURCE = "UCI Metro Interstate Traffic Volume (real data)"
except Exception as e:
    print("Could not load the UCI data ->", type(e).__name__, "| using synthetic data instead")
    df = make_synthetic()
    SOURCE = "Synthetic data (mimics the UCI pattern)"

df["date_time"] = pd.to_datetime(df["date_time"])
df["hour"] = df["date_time"].dt.hour
print("Data source:", SOURCE)
print("Rows:", len(df))
df[["date_time", "traffic_volume", "hour"]].head()

## Step 2: Derive the arrival rate from the hourly pattern

In [ ]:
hourly = df.groupby("hour")["traffic_volume"].mean()

plt.figure(figsize=(8,4))
plt.plot(hourly.index, hourly.values, marker="o")
plt.title("Average traffic volume by hour of day")
plt.xlabel("Hour"); plt.ylabel("Vehicles / hour")
plt.grid(True)
plt.savefig("results/hourly_traffic.png", dpi=150, bbox_inches="tight"); plt.show()

peak_hour = hourly.idxmax()
offpeak_hour = hourly.idxmin()
print("Peak hour:", peak_hour, "->", round(hourly[peak_hour]), "veh/h")
print("Off-peak hour:", offpeak_hour, "->", round(hourly[offpeak_hour]), "veh/h")

### Scaling to an intersection
The dataset volume is for an entire highway. We scale it to the total demand of a single intersection (`SCALE`).
Direction share: **North/South 30% each, East/West 20% each** (asymmetric, so the benefit of the adaptive controller is visible).

In [ ]:
SCALE = 0.4
SHARE = {"N": 0.30, "S": 0.30, "E": 0.20, "W": 0.20}

def arrival_rates(volume_per_hour):
    # veh/second for each approach
    return {k: volume_per_hour * SCALE * s / 3600 for k, s in SHARE.items()}

for name, h in [("Peak", peak_hour), ("Off-peak", offpeak_hour)]:
    r = arrival_rates(hourly[h])
    print(name, {k: round(v*3600) for k, v in r.items()}, "veh/h per approach")

## Step 3: Arrival model validation
We assume vehicles arrive as a **Poisson process**. We check this with:
- **Chi-square test:** whether the per-10-second arrival counts follow a Poisson distribution (Day 7)
- **KS test:** whether the inter-arrival times follow an Exponential distribution

In [ ]:
lam = arrival_rates(hourly[peak_hour])["N"]          # veh/sec, North approach
n_sec = 3600

arrivals = np.random.poisson(lam, n_sec)             # one hour, per-second arrivals
counts10 = arrivals.reshape(-1, 10).sum(axis=1)      # counts per 10-second window

# Chi-square goodness of fit (Poisson)
mu = lam * 10
values = np.arange(0, 7)
obs = np.array([(counts10 == v).sum() for v in values[:-1]] + [(counts10 >= values[-1]).sum()])
exp_p = np.append(stats.poisson.pmf(values[:-1], mu), 1 - stats.poisson.cdf(values[-2], mu)) * len(counts10)
chi2 = np.sum((obs - exp_p)**2 / exp_p)
df_chi = len(obs) - 1                                 # mu is known, so df = bins - 1
crit = stats.chi2.ppf(0.95, df_chi)
print(f"Chi-square = {chi2:.2f}, critical (0.05) = {crit:.2f}")
print("Result:", "Poisson fits (fail to reject H0)" if chi2 < crit else "Poisson does not fit (reject H0)")

# KS test for inter-arrival times
inter = np.random.exponential(1/lam, 2000)
ks_stat, ks_p = stats.kstest(inter, "expon", args=(0, 1/lam))
print(f"KS statistic = {ks_stat:.4f}, p-value = {ks_p:.4f}")
print("Result:", "Exponential fits" if ks_p > 0.05 else "Exponential does not fit")

## Step 4: Simulation model

**Model details (1-second time step):**
- 4 approaches: N, S, E, W. Each has one queue.
- 2 phases: **Phase 0 = N+S green**, **Phase 1 = E+W green**
- Saturation flow: 1 vehicle every 2 seconds (0.5 veh/s) per green approach
- Each phase change has **3 seconds of lost time** (yellow/all-red), during which no vehicle moves
- Output: average delay per vehicle, max queue, throughput

**Controllers:**
- **Fixed:** every phase gets 30 seconds of green
- **Adaptive:** min green 10s, max green 60s. After the min green, if the queue on the current phase is empty (or max green is reached) and vehicles are waiting on the other phase, the signal switches.

In [ ]:
SAT_FLOW = 0.5      # veh/sec per green approach
LOST = 3            # sec lost time at each phase change
PHASE_APPROACHES = {0: ["N", "S"], 1: ["E", "W"]}

def simulate(rates, controller="fixed", duration=3600, fixed_green=30,
             min_green=10, max_green=60, rng=None):
    rng = rng or np.random.default_rng()
    q = {k: 0 for k in rates}            # queue length
    credit = {k: 0.0 for k in rates}     # discharge credit
    phase, green_time, lost_left = 0, 0, 0
    total_delay, arrived, served, max_q = 0, 0, 0, 0

    for t in range(duration):
        # arrivals
        for k, lam in rates.items():
            a = rng.poisson(lam)
            q[k] += a; arrived += a

        if lost_left > 0:
            lost_left -= 1                      # yellow / all-red
        else:
            # discharge
            for k in PHASE_APPROACHES[phase]:
                credit[k] += SAT_FLOW
                go = min(int(credit[k]), q[k])
                q[k] -= go; served += go
                credit[k] -= int(credit[k])
            green_time += 1

            # switch decision
            cur = sum(q[k] for k in PHASE_APPROACHES[phase])
            oth = sum(q[k] for k in PHASE_APPROACHES[1 - phase])
            if controller == "fixed":
                switch = green_time >= fixed_green
            else:
                switch = (green_time >= max_green) or \
                         (green_time >= min_green and cur == 0 and oth > 0)
            if switch:
                phase = 1 - phase
                green_time, lost_left = 0, LOST
                credit = {k: 0.0 for k in credit}

        total_q = sum(q.values())
        total_delay += total_q              # vehicle-seconds
        max_q = max(max_q, max(q.values()))

    avg_delay = total_delay / max(arrived, 1)    # sec per vehicle (Little's law)
    return {"avg_delay": avg_delay, "max_queue": max_q, "throughput": served}

# quick test
r = arrival_rates(hourly[peak_hour])
print("Fixed   :", simulate(r, "fixed",    rng=np.random.default_rng(1)))
print("Adaptive:", simulate(r, "adaptive", rng=np.random.default_rng(1)))

## Step 5: Monte Carlo simulation (30 replications)
A single run gives a random result, so for each scenario we run the simulation **30 times** and take the average (as in Day 4).

In [ ]:
N_REP = 30
scenarios = {"Peak": hourly[peak_hour], "Off-peak": hourly[offpeak_hour]}
rows = []

for sc, vol in scenarios.items():
    rates = arrival_rates(vol)
    for ctrl in ["fixed", "adaptive"]:
        for i in range(N_REP):
            rng = np.random.default_rng(1000 + i)    # same seed -> fair comparison
            res = simulate(rates, ctrl, rng=rng)
            rows.append({"scenario": sc, "controller": ctrl, "rep": i, **res})

results = pd.DataFrame(rows)
summary = results.groupby(["scenario", "controller"])[["avg_delay", "max_queue", "throughput"]].mean().round(2)
summary

## Step 6: Two-sample t-test + 95% confidence interval (Day 6)

In [ ]:
ttest_rows = []
for sc in scenarios:
    fx = results[(results.scenario == sc) & (results.controller == "fixed")]["avg_delay"].values
    ad = results[(results.scenario == sc) & (results.controller == "adaptive")]["avg_delay"].values

    t_stat, p_val = stats.ttest_ind(fx, ad, equal_var=False)
    diff = fx.mean() - ad.mean()
    se = np.sqrt(fx.var(ddof=1)/len(fx) + ad.var(ddof=1)/len(ad))
    dfree = len(fx) + len(ad) - 2
    ci = stats.t.interval(0.95, dfree, loc=diff, scale=se)

    ttest_rows.append({"scenario": sc, "fixed_mean": fx.mean(), "adaptive_mean": ad.mean(), "diff": diff,
                       "t": t_stat, "p": p_val, "ci_low": ci[0], "ci_high": ci[1]})
    print(f"--- {sc} ---")
    print(f"Fixed mean delay   : {fx.mean():.2f} s")
    print(f"Adaptive mean delay: {ad.mean():.2f} s")
    print(f"Improvement        : {diff:.2f} s ({100*diff/fx.mean():.1f}%)")
    print(f"t = {t_stat:.2f}, p = {p_val:.4g}, 95% CI of diff = [{ci[0]:.2f}, {ci[1]:.2f}]")
    if p_val < 0.05:
        better = "Adaptive" if diff > 0 else "Fixed"
        print(f"Reject H0: the delays of the two controllers differ significantly. {better} is better.\n")
    else:
        print("Fail to reject H0: no significant difference.\n")

## Step 7: Graphs

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
metrics = [("avg_delay", "Avg delay (sec/vehicle)"), ("max_queue", "Max queue length"), ("throughput", "Vehicles served / hour")]
x = np.arange(len(scenarios)); w = 0.35

for a, (m, title) in zip(ax, metrics):
    f = [results[(results.scenario == s) & (results.controller == "fixed")][m].mean() for s in scenarios]
    d = [results[(results.scenario == s) & (results.controller == "adaptive")][m].mean() for s in scenarios]
    a.bar(x - w/2, f, w, label="Fixed", color="tab:red", alpha=0.8)
    a.bar(x + w/2, d, w, label="Adaptive", color="tab:green", alpha=0.8)
    a.set_xticks(x); a.set_xticklabels(list(scenarios))
    a.set_title(title); a.legend(); a.grid(True, axis="y")
plt.tight_layout()
plt.savefig("results/comparison_bars.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Box plot: delay distribution over 30 replications (peak hour)
pk = results[results.scenario == "Peak"]
plt.figure(figsize=(5, 4))
plt.boxplot([pk[pk.controller == "fixed"]["avg_delay"], pk[pk.controller == "adaptive"]["avg_delay"]])
plt.xticks([1, 2], ["Fixed", "Adaptive"])
plt.title("Peak hour: delay distribution (30 runs)")
plt.ylabel("Avg delay (sec/vehicle)"); plt.grid(True)
plt.savefig("results/boxplot_peak.png", dpi=150, bbox_inches="tight"); plt.show()

## Step 7a: Extra analysis (to strengthen the report)
**(A) 24-hour sweep:** compare Fixed vs Adaptive for every hour of the day (10 runs each).
**(B) Sensitivity:** how the benefit of the adaptive controller changes as `SCALE` (traffic load) changes (peak hour).

In [ ]:
# (A) 24-hour sweep
sweep = []
for h in range(24):
    rates = arrival_rates(hourly[h])
    for ctrl in ["fixed", "adaptive"]:
        d = [simulate(rates, ctrl, rng=np.random.default_rng(2000 + i))["avg_delay"] for i in range(10)]
        sweep.append({"hour": h, "controller": ctrl, "avg_delay": np.mean(d), "sd": np.std(d, ddof=1)})
sweep = pd.DataFrame(sweep)

plt.figure(figsize=(9, 4))
for ctrl, col in [("fixed", "tab:red"), ("adaptive", "tab:green")]:
    sub = sweep[sweep.controller == ctrl]
    plt.plot(sub.hour, sub.avg_delay, marker="o", label=ctrl.capitalize(), color=col)
    plt.fill_between(sub.hour, sub.avg_delay - sub.sd, sub.avg_delay + sub.sd, color=col, alpha=0.15)
plt.title("Average delay by hour of day (10 runs per point, band = +-1 SD)")
plt.xlabel("Hour of day"); plt.ylabel("Avg delay (sec/vehicle)"); plt.xticks(range(0, 24, 2))
plt.legend(); plt.grid(True)
plt.savefig("results/delay_by_hour.png", dpi=150, bbox_inches="tight"); plt.show()

# (B) Sensitivity to load
sens = []
base_vol = hourly[peak_hour]
for sc in [0.2, 0.3, 0.4, 0.5, 0.6]:
    rates = {k: base_vol * sc * sh / 3600 for k, sh in SHARE.items()}
    for ctrl in ["fixed", "adaptive"]:
        d = [simulate(rates, ctrl, rng=np.random.default_rng(3000 + i))["avg_delay"] for i in range(10)]
        sens.append({"SCALE": sc, "controller": ctrl, "avg_delay": np.mean(d)})
sens = pd.DataFrame(sens)
sens_pivot = sens.pivot(index="SCALE", columns="controller", values="avg_delay").round(2)
sens_pivot["improvement_%"] = (100 * (sens_pivot["fixed"] - sens_pivot["adaptive"]) / sens_pivot["fixed"]).round(1)
sens_pivot

## Step 7b: Save results and zip
Running this cell creates `traffic_results.zip` (data, summary, t-test, and all graphs). In Colab it downloads automatically.

In [ ]:
import shutil
results.to_csv("results/all_runs.csv", index=False)
summary.to_csv("results/summary.csv")
sweep.round(3).to_csv("results/sweep_24h.csv", index=False)
sens_pivot.to_csv("results/sensitivity.csv")
pd.DataFrame(ttest_rows).round(4).to_csv("results/ttest_results.csv", index=False)
with open("results/info.txt", "w") as f:
    f.write(f"Data source: {SOURCE}\nRows: {len(df)}\nPeak hour: {peak_hour}\nOff-peak hour: {offpeak_hour}\n")
    f.write(f"SCALE={SCALE}, SHARE={SHARE}, N_REP={N_REP}\n")
    f.write(f"Chi-square={chi2:.2f} (critical {crit:.2f}), KS p={ks_p:.4f}\n")

shutil.make_archive("traffic_results", "zip", "results")
print("Created: traffic_results.zip ->", sorted(os.listdir("results")))

try:
    from google.colab import files
    files.download("traffic_results.zip")
except ImportError:
    print("Not running in Colab, so just take traffic_results.zip from the working folder.")

## Step 8: Conclusion (write according to your own results)
- During the peak hour, the Adaptive signal can reduce average delay compared with the Fixed signal, because it gives more green time to the direction with more vehicles.
- Off-peak, traffic is light, so the difference between the two controllers may be small.
- If the t-test p-value is below 0.05, the difference is **statistically significant**.

**Limitations:** single lane, no turning movements, constant lambda within each hour, and driver behaviour is ignored.

**Future work:** multiple intersections, a left-turn phase, emergency vehicle priority, and real Dhaka data.

### Student Work
1. Change `SCALE` to 0.3 and 0.5 and compare the results. At which value is the adaptive controller more beneficial?
2. Change `min_green` and `max_green` to find the best values.
3. Change `SHARE` so that East/West is busier. What happens?